In [0]:
import re

# Configuration
VOLUME_PATH = "/Volumes/smart_meters/bronze/tad_volume"

# ==========================================
# 1. INGESTÃO DO DATASET DE CONSUMO (hhblock)
# ==========================================
consumption_path = f"{VOLUME_PATH}/hhblock_dataset/*.csv"
table_consumption_path = f"smart_meters.bronze.consumption"

try:
    df_consumption = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(consumption_path)

    # Gravar na Bronze
    df_consumption.write.mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_consumption_path)
    
    print(f"✅ Tabela de consumo guardada em {table_consumption_path}! (Linhas lidas: {df_consumption.count()})")
except Exception as e:
    print(f"❌ Erro ao carregar o consumo: {e}")


# ==========================================
# 2. INGESTÃO DAS TABELAS DE REFERÊNCIA
# ==========================================
reference_files = {
    "households": ("informations_households.csv", ","),
    "weather": ("weather_hourly_darksky.csv", ","),
    "tariffs": ("Tariffs.csv", ";"),
    "holidays": ("uk_bank_holidays.csv", ",")
}

for table_name, (file_name, dlmtr) in reference_files.items():
    file_path = f"{VOLUME_PATH}/{file_name}"
    
    try:
        df_spark = spark.read.format("csv") \
            .option("header", "true") \
            .option("sep", dlmtr) \
            .option("inferSchema", "true") \
            .load(file_path)
        
        # Limpeza de nomes de colunas
        for col_name in df_spark.columns:
            clean_name = re.sub(r'[ ,;{}()\n\t=]+', '_', col_name.strip())
            df_spark = df_spark.withColumnRenamed(col_name, clean_name)
        
        table_path = f"smart_meters.bronze.{table_name}"
        df_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_path)
        
        print(f"✅ Tabela '{table_path}' criada com colunas higienizadas!")
        
    except Exception as e:
        print(f"❌ Erro ao carregar {file_name}: {e}")

In [0]:
%sql
TRUNCATE TABLE smart_meters.bronze.dim_tariffs;

MERGE INTO smart_meters.bronze.dim_tariffs AS target
USING (
  SELECT DISTINCT
    col1 AS tariff_id,
    col2 AS type,
    col3 AS period,
    col4 AS price
  FROM VALUES 
    (1, 'Std', 'N/A', 0.14),
    (2, 'ToU', 'Low', 0.04),
    (3, 'ToU', 'Normal', 0.12),
    (4, 'ToU', 'High', 0.67)
) AS source
ON target.tariff_id = source.tariff_id

WHEN MATCHED THEN 
  UPDATE SET 
    target.type = source.type,
    target.period = source.period,
    target.price = source.price

WHEN NOT MATCHED THEN 
  INSERT *;

SELECT * FROM smart_meters.bronze.dim_tariffs;

In [0]:
%sql

SELECT COUNT(*) AS row_count
FROM smart_meters.bronze.consumption;

In [0]:
%sql

SELECT COUNT(*) AS row_count
FROM smart_meters.bronze.consumption;

In [0]:
%sql

SELECT
  LCLid,
  day,
  hh_0,
  hh_1,
  hh_2,
  hh_3,
  hh_4,
  hh_5,
  hh_6
FROM
  smart_meters.bronze.consumption
WHERE
    LCLid = 'MAC000331'